# CMPU395 Final Project Analysis

Jensen Bergman and Jacob Sellitti - 5/1/26

**PUT CODE BLOCKS FROM OTHER FILES BETWEEN EACH OF THESE WRITE-UP SECTIONS**

## Introduction

Constructing an optimal batting order is a complex problem that blends player performance, team context, and strategic decision-making. In this notebook, we take a data-driven approach to this problem using Statcast data from the 2025 MLB season.

Our goal is to model how players are assigned lineup positions based on their offensive performance up to that point in the season. To do this, we implement a two-model framework:

- A **player-level model** that predicts the probability distribution over lineup positions (1–9) for an individual player
- A **team-level model** that predicts a full lineup for a group of nine players


## Data Collection

Using the pybaseball library, we scraped pitch-by-pitch Statcast data for the 2025 MLB season. This dataset contains detailed information about every pitch, including outcomes (hits, strikeouts, walks), batted ball metrics (launch angle, exit velocity), and game context.

In addition, we incorporate lineup data scraped separately, which provides the batting order for each team in each game. This allows us to map individual player performances to their corresponding lineup positions.

The raw Statcast dataset is highly granular (one row per pitch), so a key challenge is transforming this into meaningful player-level features that evolve over the course of a season.


## Data Preparation and Feature Engineering

To construct meaningful inputs for our models, we aggregate pitch-level data into player-level statistics that reflect performance up to/before each game. This prevents data leakage and better reflects real-world decision-making.

For each player, we compute the following cumulative statistics:
- Batting Average (AVG) - Fraction of hits/at-bats of an individual player. This will be between 0 and 1.
- Slugging Percentage (SLG) - Fraction of total bases/at bats of an individual player, where singles are worth 1, doubles are worth 2, triples are worth 3, and home runs are worth 4. This will be between 0 and 4.
- On base percentage (OBP) - Fraction of times on base (hits + walks + hit by pitches) / plate appearances. Will be between 0 and 1.
- Expected weighted on-base average (xwOBA) - Measures offensive performance based on quality of contact (exit velocity and launch angle), removing the effect of defense and luck.
- Home Runs (HR) - Total number of home runs a batter has hit.
- Runs Batted In (RBI) - Total number of runs a batter has batted in.
- Stolen bases (SB) - Total number of stolen bases a player has.
- Contact percentage - Fraction of total contact (balls put in play) / total number of swings.
- Strikeout percentage (K%) - Fraction of strikeouts/total at-bats.
- Walk percentage (BB%) - Fraction of walks/total at-bats.
- Strikeout/Walk Ratio (K/BB) - Ratio of strikeouts to walks
- Sweet spot percentage - The percentage of the time a hitter hits a ball with a launch angle between 8 and 32 degrees.
- Hard hit percentage (HH%) - The percentage of the time a hitter hits a ball over 95 MPH.

We also implement safeguards such as:
- Safe division to avoid undefined values
- Filtering out early-season data (first 10 games) to allow statistics to stabilize
- Removing pinch hitters (players without consistent lineup positions) to ensure we have a max of 9 players per team per game

Finally, we join this data with lineup information to assign each player their batting order position for each game, and save the completed feature data.


## Dataset Construction

After feature engineering, we construct a dataset where each row represents a **player-game instance**, including:
- Player ID
- Team
- Game date
- Performance features
- Observed lineup position (target variable)

To ensure realistic evaluation, we split the data using a grouped train-test split, where all players from a single game are kept together. This prevents leakage between training and validation sets and ensures that full lineups are only seen in one split.

We also normalize features using standard scaling to improve model training stability.


## Exploratory Data Analysis

Before training models, we explore the relationships between player performance metrics and lineup positions.

We expect to see patterns such as:
- Higher OBP players appearing earlier in the lineup
- Power hitters (high SLG, HR) appearing in middle positions
- Lower-contact hitters appearing later in the lineup

These patterns help validate that the dataset captures meaningful signals and provides intuition for how the model may learn lineup assignments.


## Player-Level Model

We first train a neural network to predict lineup position for individual players.

This model takes in a vector of player performance features and outputs a probability distribution over the nine lineup positions using a softmax layer.

The architecture consists of:
- Two fully connected layers
- A learned embedding layer (32-dimensional representation)
- A final output layer with 9 logits

We train the model using categorical cross-entropy loss and the Adam optimizer, with L2 regularization to prevent overfitting. This model allows us to capture uncertainty in lineup placement, as players may occupy different positions throughout the season.

We evaluate the player model using several metrics:

- Top-1 Accuracy: Exact prediction of lineup position
- Top-3 Accuracy: Whether the true position is among the top 3 predicted probabilities
- Mean Absolute Error (MAE): Average positional distance between prediction and truth

Top-3 accuracy is particularly important, as lineup positions are ordinal and small errors (e.g., predicting 2 instead of 1) are less severe than larger ones. We also analyze prediction outputs and save them for further use in downstream modeling.

## Team-Level Model

While the player model predicts positions independently, lineup construction is inherently a joint problem involving all nine players. To address this, we build a team-level model that takes all nine players in a lineup as input and predicts their positions simultaneously.

This model leverages the embeddings learned by the player model, using them as input features. By doing so, it captures richer representations of player skill while modeling interactions between players.

The model outputs a matrix of scores representing each player’s suitability for each lineup position.

To convert model outputs into a valid lineup (each position assigned exactly once), we apply the Hungarian algorithm (linear sum assignment). This ensures:
- Each player is assigned exactly one position
- Each lineup position is filled exactly once
- The overall assignment maximizes model confidence

We evaluate the team model using several metrics:
- Exact Match Accuracy: Percentage of perfectly predicted lineups
- Average Displacement: Average positional error across players
- Pairwise Accuracy: How often the model correctly predicts which player bats before another

We analyze position-specific accuracy and construct a confusion matrix to understand common prediction errors, and also use permutation importance to understand which features drive predictions.

## Results, Insights, and Conclusions

Our results show that lineup position can be predicted with meaningful accuracy using player performance metrics.

Key observations include:
- Certain positions (e.g., beginning of the lineup) are easier to predict than others
- The model captures general lineup trends
- Joint modeling improves coherence of predicted lineups compared to independent predictions

However, performance is limited by factors not captured in the data, such as managerial preferences, injuries, and game-specific strategy. This work highlights both the potential and limitations of data-driven approaches to baseball strategy, and provides a foundation for future improvements incorporating additional context and data.

